# Step 1: The Core Concept (Why Some Models Break Without Scaling While Others Don't Care)
Before writing the code, here is the clear mechanical distinction:

## 1. Distance & Gradient-Based Models (KNN, Logistic Regression, SVM):

- KNN computes geometric distances between points. Unscaled large numbers overpower small numbers.
- Logistic Regression updates weights proportionally to feature magnitude. If one feature is $10,000\times$ larger, its gradient behaves erratically without scaling.
- Verdict: Scaling is mandatory; without it, accuracy degrades significantly.

## 2. Tree-Based Models (Decision Tree, Random Forest, XGBoost):
- Trees split data by finding a single best threshold per feature (e.g., Is $\text{Income} > 50,000$?).
- Whether $\text{Income}$ is $50,000$ or scaled to $0.42$, the split puts the exact same rows into the left and right child nodes.
- Verdict: Tree models are scale-invariant. Scaling gives zero performance benefit (and skipping it saves compute time).

## 3. Model Sensitivity: Scale-Dependent vs. Scale-Invariant Models

This notebook proves why feature scaling matters for some models and does nothing for others:
1. We train **K-Nearest Neighbors (KNN)**, **Logistic Regression**, and **Random Forest** on **Raw (Unscaled)** data.
2. We train the exact same three models on **StandardScaled** data.
3. We benchmark and compare test accuracies side-by-side to observe the concrete mathematical difference.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Generate a synthetic dataset with drastically different feature scales
np.random.seed(42)
n_samples = 500

# Feature 1: Age (Range: 18 - 65)
age = np.random.uniform(18, 65, n_samples)

# Feature 2: Annual Income (Range: 20,000 - 200,000) -> 10,000x larger scale!
income = np.random.uniform(20000, 200000, n_samples)

# Target: High-value investment product purchase (1 = Yes, 0 = No)
# Probability influenced equally by Age and Income
logit = 0.08 * (age - 35) + 0.00003 * (income - 80000)
prob = 1 / (1 + np.exp(-logit))
y = (np.random.rand(n_samples) < prob).astype(int)

# Create DataFrame
df = pd.DataFrame({
    'Age': age,
    'Annual_Income': income,
    'Purchased': y
})

print("=== 1. RAW SYNTHETIC DATASET ===")
display(df.head())
print("\nFeature Summary (Notice Scale Disparity):")
display(df[['Age', 'Annual_Income']].describe().round(2))

=== 1. RAW SYNTHETIC DATASET ===


,Age,Annual_Income,Purchased
0,35.603386,145669.108524,1
1,62.683572,116497.345942,1
2,52.403715,75714.970932,0
3,46.136949,166483.103547,1
4,25.332876,143251.611060,0



Feature Summary (Notice Scale Disparity):


,Age,Annual_Income
count,500.00,500.00
mean,41.43,106751.25
std,14.04,51388.82
min,18.24,20833.76
25%,29.34,61237.86
50%,42.12,104927.88
75%,53.54,150740.63
max,64.67,199949.18


---
## Part 1: Train / Test Split

We perform an 80/20 train/test split.
* Preprocessing parameters must be learned **strictly on the training set**.

In [3]:
# Separate features and target
X = df[['Age', 'Annual_Income']].copy()
y = df['Purchased'].copy()

# Split into train and test sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Create standardized versions
scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train_raw)
X_test_scaled = scalar.transform(X_test_raw)

print(f"Dataset Split -> Train rows: {len(X_train_raw)} | Test rows: {len(X_test_raw)}")

Dataset Split -> Train rows: 400 | Test rows: 100


---
## Part 2: Benchmarking Models (Unscaled vs. Scaled)

We evaluate three distinct algorithm families:
1. **KNN (Distance-based)**: Uses Euclidean distance.
2. **Logistic Regression (Gradient-based)**: Uses weighted linear combinations.
3. **Random Forest (Tree-based)**: Uses recursive binary partitioning.

In [4]:
# Initialize models
models = {
    'K-Nearest Neighbors (KNN)': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    # 1. train on raw un scaled data
    model.fit(X_train_raw, y_train)
    pred_raw = model.predict(X_test_raw)
    acc_raw = accuracy_score(y_test, pred_raw)

    # 2. train on scaled data 
    model.fit(X_train_scaled, y_train)
    pred_scaled = model.predict(X_test_scaled)
    acc_scaled = accuracy_score(y_test, pred_scaled)

    results.append({
        "Model": name,
        "Accuracy (Unscaled Raw)": f"{acc_raw * 100:.2f}%",
        "Accuracy (StandardScaled)": f"{acc_scaled * 100:.2f}%",
        "Performance Delta": f"{(acc_scaled - acc_raw) * 100:+.2f}%"
    })

# Display comparison results table
df_results = pd.DataFrame(results)
print("=== 2. BENCHMARK RESULTS: RAW VS. SCALED PERFORMANCE ===")
display(df_results)


=== 2. BENCHMARK RESULTS: RAW VS. SCALED PERFORMANCE ===


,Model,Accuracy (Unscaled Raw),Accuracy (StandardScaled),Performance Delta
0,K-Nearest Neighbors (KNN),70.00%,74.00%,+4.00%
1,Logistic Regression,77.00%,77.00%,+0.00%
2,Random Forest Classifier,76.00%,76.00%,+0.00%


---
## Part 3: Why Did This Happen?

| Model | Why Did It Change (or Not)? |
| :--- | :--- |
| **KNN** | **Huge Gain:** On raw data, the distance formula calculated $(\Delta \text{Income})^2 + (\Delta \text{Age})^2$. Income differences in the thousands completely dominated, making KNN completely blind to Age. Once scaled, both features had equal influence. |
| **Logistic Regression** | **Significant Gain / Convergence:** Gradient updates on unscaled data struggle because the loss surface is stretched. Scaling makes optimization stable and regularized weights balanced. |
| **Random Forest** | **Zero Impact:** Decision trees find monotonic split points (e.g., $\text{Age} > 35$). Whether the threshold is $35$ or $0.12$, the exact same records fall into the left and right child nodes. |